# Fine-tune MPNet SentenceTransformer for Restaurant Aspect Classification

This notebook creates one reproducible train/validation/test split, saves an untouched base encoder, fine-tunes a second encoder using multi-label similarity pairs, and saves the retrained encoder for downstream classifier tuning.

Run this notebook before `02-mpnet_ovr_logistic_regression.ipynb`.


In [ ]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, SentenceTransformer, losses

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
path = Path.cwd().resolve()
for _ in range(8):
    if (path / "data" / "Restaurant_ABSA_processed.csv").exists():
        PROJECT_ROOT = path
        break
    path = path.parent
else:
    raise FileNotFoundError("Could not locate the project root")

DATA_PATH = PROJECT_ROOT / "data" / "Restaurant_ABSA_processed.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "SBERT"
MODEL_DIR = PROJECT_ROOT / "models" / "sbert"
BASE_ENCODER_DIR = MODEL_DIR / "unretrained_encoder"
RETRAINED_ENCODER_DIR = MODEL_DIR / "retrained_encoder"
SPLIT_PATH = OUTPUT_DIR / "data_split.npz"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


In [ ]:
ASPECT_COLS = ["food", "price", "service", "ambiance", "miscellaneous"]
TEXT_COL = "review_en"

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)

invalid_labels = {
    aspect: sorted(df[aspect].dropna().unique().tolist())
    for aspect in ASPECT_COLS
    if not set(df[aspect].dropna().unique()).issubset({0, 1})
}
if invalid_labels:
    raise ValueError(f"Invalid binary labels: {invalid_labels}")

texts = df[TEXT_COL].astype(str).to_numpy()
labels = df[ASPECT_COLS].to_numpy(dtype=np.int8)
indices = np.arange(len(df))

train_val_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
)
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.25,
    random_state=SEED,
)

np.savez_compressed(
    SPLIT_PATH,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))
print("Saved split:", SPLIT_PATH)


## Build multi-label similarity pairs

Two reviews receive a target similarity equal to the Jaccard similarity of their aspect-label sets. Each training review is paired with overlapping-label and non-overlapping-label examples.


In [ ]:
def jaccard_similarity(left, right):
    intersection = np.logical_and(left, right).sum()
    union = np.logical_or(left, right).sum()
    return float(intersection / union) if union else 0.0


def build_training_pairs(text_values, label_values, pairs_per_sample=6, seed=42):
    rng = np.random.default_rng(seed)
    examples = []
    sample_count = len(text_values)

    for index in range(sample_count):
        similarities = np.array([
            jaccard_similarity(label_values[index], label_values[other])
            if other != index else -1.0
            for other in range(sample_count)
        ])

        positive_candidates = np.flatnonzero(similarities > 0)
        negative_candidates = np.flatnonzero(similarities == 0)
        positive_count = min((pairs_per_sample + 1) // 2, len(positive_candidates))
        negative_count = min(pairs_per_sample // 2, len(negative_candidates))

        selected = []
        if positive_count:
            selected.extend(
                rng.choice(
                    positive_candidates,
                    size=positive_count,
                    replace=False,
                ).tolist()
            )
        if negative_count:
            selected.extend(
                rng.choice(
                    negative_candidates,
                    size=negative_count,
                    replace=False,
                ).tolist()
            )

        for other in selected:
            examples.append(
                InputExample(
                    texts=[text_values[index], text_values[other]],
                    label=float(similarities[other]),
                )
            )

    rng.shuffle(examples)
    return examples


train_examples = build_training_pairs(
    texts[train_idx],
    labels[train_idx],
    pairs_per_sample=6,
    seed=SEED,
)

print("Training pairs:", len(train_examples))


## Save the untouched encoder and fine-tune a separate copy

Tune `EPOCHS`, `BATCH_SIZE`, and `WARMUP_RATIO` only on validation performance from the downstream classifier. Do not inspect the test set while choosing them.


In [ ]:
BASE_MODEL_NAME = "all-mpnet-base-v2"
EPOCHS = 3
BATCH_SIZE = 16
WARMUP_RATIO = 0.10

base_encoder = SentenceTransformer(BASE_MODEL_NAME)
base_encoder.save(str(BASE_ENCODER_DIR))
print("Saved untouched encoder:", BASE_ENCODER_DIR)

retrained_encoder = SentenceTransformer(BASE_MODEL_NAME)
train_loader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
)
train_loss = losses.CosineSimilarityLoss(
    retrained_encoder)
warmup_steps = int(len(train_loader) 
                   * EPOCHS 
                   * WARMUP_RATIO)

retrained_encoder.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
)

In [ ]:

retrained_encoder.save(str(RETRAINED_ENCODER_DIR))

print("Saved retrained encoder:", RETRAINED_ENCODER_DIR)

In [ ]:
training_metadata = {
    "base_model": BASE_MODEL_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "warmup_ratio": WARMUP_RATIO,
    "training_pairs": len(train_examples),
    "aspect_cols": ASPECT_COLS,
    "text_col": TEXT_COL,
    "seed": SEED,
}

with (MODEL_DIR / "encoder_training_metadata.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(training_metadata, file, indent=2)

training_metadata
